# Experimentación: Gráficos de Cleveland

Este notebook te permite experimentar con gráficos de Cleveland (dot plots) para comparaciones.

**Instrucciones:**
1. Ejecuta las primeras celdas para cargar datos
2. Ve a la sección "EXPERIMENTA AQUÍ"
3. Cambia solo los parámetros marcados
4. Ejecuta y observa resultados

## 1. Importaciones y Configuración

In [ ]:
# Importaciones.
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Agregar rutas.
Ruta_Base = Path.cwd().parent.parent
sys.path.append(str(Ruta_Base / "Codigo" / "Utilidades"))
sys.path.append(str(Ruta_Base / "Codigo" / "Visualizacion"))

from Configuracion import *
from Graficos_Cleveland import *

print("✓ Módulos importados correctamente")

## 2. Cargar Datos

In [ ]:
# Cargar datos procesados.
Ruta_Generales = RUTA_BASES_DEFINITIVAS / "Bases finalesGenerales.xlsx"
Ruta_Ballotage = RUTA_BASES_DEFINITIVAS / "Bases finalesBallotage.xlsx"

# Si los archivos existen, cargarlos.
if Ruta_Generales.exists():
    Df_Generales = pd.read_excel(Ruta_Generales)
    print(f"✓ Generales cargado: {len(Df_Generales)} filas")
else:
    print("⚠️ Archivo Generales no encontrado")
    # Crear datos de ejemplo.
    np.random.seed(42)
    Df_Generales = pd.DataFrame({
        'ID': range(1000),
        'Categoria_PASO_2023': np.random.choice(
            ['Left_Wing', 'Progressivism', 'Centre', 
             'Moderate_Right_A', 'Right_Wing_Libertarian'],
            1000
        ),
        'CO_Item_3_Izq': np.random.randn(1000) * 0.5,
        'CO_Item_3_Der': np.random.randn(1000) * 0.5,
        'CT_Item_3_Izq': np.random.randn(1000) * 100 + 50,
        'CT_Item_3_Der': np.random.randn(1000) * 100 + 50
    })
    print("✓ Usando datos de ejemplo")

if Ruta_Ballotage.exists():
    Df_Ballotage = pd.read_excel(Ruta_Ballotage)
    print(f"✓ Ballotage cargado: {len(Df_Ballotage)} filas")
else:
    print("⚠️ Archivo Ballotage no encontrado")
    Df_Ballotage = pd.DataFrame({
        'ID': range(500),
        'Categoria_PASO_2023': np.random.choice(
            ['Left_Wing', 'Progressivism', 'Right_Wing_Libertarian'],
            500
        ),
        'CO_Item_3_Izq': np.random.randn(500) * 0.5,
        'CO_Item_3_Der': np.random.randn(500) * 0.5
    })
    print("✓ Usando datos de ejemplo")

---
# EXPERIMENTA AQUÍ

## Cleveland Básico: Comparación Izquierda vs Derecha

**Cambia estos parámetros:**

In [ ]:
# ============================================================
# PARÁMETROS PARA EXPERIMENTAR
# ============================================================

# Variables a comparar (cambio de opinión por item).
Items_Comparar = [
    'Item_3',
    'Item_4',
    'Item_5',
    'Item_6',
    'Item_7'
]

# Grupos a comparar.
Grupos_Izquierda = ['Left_Wing', 'Progressivism']
Grupos_Derecha = ['Moderate_Right_A', 'Right_Wing_Libertarian']

# Tipo de variable (CO o CT).
Tipo_Variable = 'CO'  # Cambia a 'CT' para tiempos

# Dirección política (Izq o Der).
Direccion = 'Izq'  # Cambia a 'Der' para derecha

# Título personalizado.
Titulo = f'Comparación {Tipo_Variable} {Direccion}: Izquierda vs Derecha'

# Tamaño de figura.
Tamano_Figura = (10, 6)  # Prueba: (12, 8), (8, 5), etc.

# ============================================================
# PREPARAR DATOS
# ============================================================

# Calcular medianas por grupo para cada item.
Datos_Grafico = []

for Item in Items_Comparar:
    Variable = f'{Tipo_Variable}_{Item}_{Direccion}'
    
    if Variable in Df_Generales.columns:
        # Mediana izquierda.
        Mediana_Izq = Df_Generales[
            Df_Generales['Categoria_PASO_2023'].isin(Grupos_Izquierda)
        ][Variable].median()
        
        # Mediana derecha.
        Mediana_Der = Df_Generales[
            Df_Generales['Categoria_PASO_2023'].isin(Grupos_Derecha)
        ][Variable].median()
        
        Datos_Grafico.append({
            'Item': Item,
            'Izquierda': Mediana_Izq,
            'Derecha': Mediana_Der,
            'Diferencia': Mediana_Izq - Mediana_Der
        })

Df_Comparacion = pd.DataFrame(Datos_Grafico)

print(f"\n✓ Datos preparados para {len(Df_Comparacion)} items")
print(f"\nPrimeras filas:")
print(Df_Comparacion.head())

In [ ]:
# ============================================================
# CREAR GRÁFICO DE CLEVELAND
# ============================================================

Fig, Ax = plt.subplots(figsize=Tamano_Figura)

# Ordenar por diferencia para visualización más clara.
Df_Ordenado = Df_Comparacion.sort_values('Diferencia')

# Posiciones verticales.
Y_Pos = np.arange(len(Df_Ordenado))

# Líneas conectando izquierda y derecha.
for i, Fila in Df_Ordenado.iterrows():
    Ax.plot(
        [Fila['Izquierda'], Fila['Derecha']],
        [Y_Pos[Df_Ordenado.index.get_loc(i)]] * 2,
        color='gray',
        linewidth=1.5,
        alpha=0.6,
        zorder=1
    )

# Puntos para izquierda.
Ax.scatter(
    Df_Ordenado['Izquierda'],
    Y_Pos,
    color='blue',
    s=100,
    label='Izquierda',
    zorder=2,
    edgecolors='black',
    linewidths=1
)

# Puntos para derecha.
Ax.scatter(
    Df_Ordenado['Derecha'],
    Y_Pos,
    color='red',
    s=100,
    label='Derecha',
    zorder=2,
    edgecolors='black',
    linewidths=1
)

# Configuración de ejes.
Ax.set_yticks(Y_Pos)
Ax.set_yticklabels(Df_Ordenado['Item'])
Ax.set_xlabel(f'Mediana {Tipo_Variable}', fontsize=12)
Ax.set_ylabel('Item', fontsize=12)
Ax.set_title(Titulo, fontsize=14, fontweight='bold')
Ax.legend(loc='best', frameon=True, shadow=True)
Ax.grid(axis='x', alpha=0.3, linestyle='--')
Ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\n✓ Gráfico de Cleveland generado")

## Cleveland: Comparación Congruentes vs Incongruentes

**Cambia estos parámetros:**

In [ ]:
# ============================================================
# PARÁMETROS PARA EXPERIMENTAR
# ============================================================

# Candidatos a analizar.
Candidatos_Analizar = [
    'Milei',
    'Bullrich',
    'Bregman',
    'Solano'
]

# Tipo de cambio.
Tipo_Cambio = 'CO'  # Cambia a 'CT' para tiempos

# Título personalizado.
Titulo_Congruencia = (
    f'Comparación {Tipo_Cambio} Congruentes vs Incongruentes'
)

# ============================================================
# PREPARAR DATOS
# ============================================================

# Verificar si existen columnas de congruencia.
if f'{Tipo_Cambio}_Congruentes_Promedio' in Df_Generales.columns:
    # Calcular medianas por candidato.
    Datos_Congruencia = []
    
    for Candidato in Candidatos_Analizar:
        # Filtrar por candidato si hay columna.
        # (Aquí asumimos que hay una columna que indica candidato)
        
        Mediana_Cong = Df_Generales[
            f'{Tipo_Cambio}_Congruentes_Promedio'
        ].median()
        
        Mediana_Incong = Df_Generales[
            f'{Tipo_Cambio}_Incongruentes_Promedio'
        ].median()
        
        Datos_Congruencia.append({
            'Candidato': Candidato,
            'Congruente': Mediana_Cong,
            'Incongruente': Mediana_Incong,
            'Diferencia': Mediana_Cong - Mediana_Incong
        })
    
    Df_Congruencia = pd.DataFrame(Datos_Congruencia)
    print(f"\n✓ Datos de congruencia preparados")
    print(Df_Congruencia)
else:
    print(
        f"\n⚠️ Columnas de congruencia no encontradas. "
        f"Creando datos de ejemplo."
    )
    Df_Congruencia = pd.DataFrame({
        'Candidato': Candidatos_Analizar,
        'Congruente': np.random.randn(len(Candidatos_Analizar)) * 0.3 + 0.5,
        'Incongruente': np.random.randn(len(Candidatos_Analizar)) * 0.3 - 0.2,
        'Diferencia': np.random.randn(len(Candidatos_Analizar)) * 0.5
    })
    print(Df_Congruencia)

In [ ]:
# ============================================================
# CREAR GRÁFICO DE CLEVELAND - CONGRUENCIA
# ============================================================

Fig, Ax = plt.subplots(figsize=(10, 6))

# Ordenar por diferencia.
Df_Cong_Ordenado = Df_Congruencia.sort_values('Diferencia')

# Posiciones verticales.
Y_Pos_Cong = np.arange(len(Df_Cong_Ordenado))

# Líneas conectando congruente e incongruente.
for i, Fila in Df_Cong_Ordenado.iterrows():
    Ax.plot(
        [Fila['Congruente'], Fila['Incongruente']],
        [Y_Pos_Cong[Df_Cong_Ordenado.index.get_loc(i)]] * 2,
        color='gray',
        linewidth=2,
        alpha=0.6,
        zorder=1
    )

# Puntos para congruente.
Ax.scatter(
    Df_Cong_Ordenado['Congruente'],
    Y_Pos_Cong,
    color='green',
    s=150,
    label='Congruente',
    zorder=2,
    edgecolors='black',
    linewidths=1.5
)

# Puntos para incongruente.
Ax.scatter(
    Df_Cong_Ordenado['Incongruente'],
    Y_Pos_Cong,
    color='orange',
    s=150,
    label='Incongruente',
    zorder=2,
    edgecolors='black',
    linewidths=1.5
)

# Configuración de ejes.
Ax.set_yticks(Y_Pos_Cong)
Ax.set_yticklabels(Df_Cong_Ordenado['Candidato'])
Ax.set_xlabel(f'Mediana {Tipo_Cambio}', fontsize=12)
Ax.set_ylabel('Candidato', fontsize=12)
Ax.set_title(Titulo_Congruencia, fontsize=14, fontweight='bold')
Ax.legend(loc='best', frameon=True, shadow=True)
Ax.grid(axis='x', alpha=0.3, linestyle='--')
Ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\n✓ Gráfico de congruencia generado")

## Cleveland: Comparación Generales vs Ballotage

**Cambia estos parámetros:**

In [ ]:
# ============================================================
# PARÁMETROS PARA EXPERIMENTAR
# ============================================================

# Items a comparar entre elecciones.
Items_Elecciones = [
    'Item_3',
    'Item_4',
    'Item_5'
]

# Tipo y dirección.
Tipo_Elecciones = 'CO'
Direccion_Elecciones = 'Izq'

# ============================================================
# PREPARAR DATOS
# ============================================================

Datos_Elecciones = []

for Item in Items_Elecciones:
    Variable = f'{Tipo_Elecciones}_{Item}_{Direccion_Elecciones}'
    
    if Variable in Df_Generales.columns:
        Mediana_Gen = Df_Generales[Variable].median()
    else:
        Mediana_Gen = np.random.randn() * 0.3
    
    if Variable in Df_Ballotage.columns:
        Mediana_Bal = Df_Ballotage[Variable].median()
    else:
        Mediana_Bal = np.random.randn() * 0.3
    
    Datos_Elecciones.append({
        'Item': Item,
        'Generales': Mediana_Gen,
        'Ballotage': Mediana_Bal,
        'Diferencia': Mediana_Gen - Mediana_Bal
    })

Df_Elecciones = pd.DataFrame(Datos_Elecciones)
print(f"\n✓ Datos de elecciones preparados")
print(Df_Elecciones)

In [ ]:
# ============================================================
# CREAR GRÁFICO DE CLEVELAND - ELECCIONES
# ============================================================

Fig, Ax = plt.subplots(figsize=(10, 6))

# Ordenar por diferencia.
Df_Elec_Ordenado = Df_Elecciones.sort_values('Diferencia')

# Posiciones verticales.
Y_Pos_Elec = np.arange(len(Df_Elec_Ordenado))

# Líneas.
for i, Fila in Df_Elec_Ordenado.iterrows():
    Ax.plot(
        [Fila['Generales'], Fila['Ballotage']],
        [Y_Pos_Elec[Df_Elec_Ordenado.index.get_loc(i)]] * 2,
        color='gray',
        linewidth=2,
        alpha=0.6,
        zorder=1
    )

# Puntos Generales.
Ax.scatter(
    Df_Elec_Ordenado['Generales'],
    Y_Pos_Elec,
    color='purple',
    s=150,
    label='Generales',
    zorder=2,
    edgecolors='black',
    linewidths=1.5
)

# Puntos Ballotage.
Ax.scatter(
    Df_Elec_Ordenado['Ballotage'],
    Y_Pos_Elec,
    color='cyan',
    s=150,
    label='Ballotage',
    zorder=2,
    edgecolors='black',
    linewidths=1.5
)

# Configuración.
Ax.set_yticks(Y_Pos_Elec)
Ax.set_yticklabels(Df_Elec_Ordenado['Item'])
Ax.set_xlabel(f'Mediana {Tipo_Elecciones}', fontsize=12)
Ax.set_ylabel('Item', fontsize=12)
Ax.set_title(
    f'Comparación {Tipo_Elecciones} {Direccion_Elecciones}: '
    f'Generales vs Ballotage',
    fontsize=14,
    fontweight='bold'
)
Ax.legend(loc='best', frameon=True, shadow=True)
Ax.grid(axis='x', alpha=0.3, linestyle='--')
Ax.axvline(0, color='black', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

print(f"\n✓ Gráfico de elecciones generado")

---
## Guardar Gráfico

Cuando tengas un gráfico que te guste, guárdalo así:

In [ ]:
# Definir ruta de salida.
Ruta_Guardado = (
    RUTA_GRAFICOS /
    "Cleveland" /
    "Mi_Cleveland.png"
)

# Crear carpeta si no existe.
Ruta_Guardado.parent.mkdir(parents=True, exist_ok=True)

# Guardar el gráfico actual.
Fig.savefig(
    Ruta_Guardado,
    dpi=300,
    bbox_inches='tight'
)

print(f"✓ Gráfico guardado en: {Ruta_Guardado}")